# Angle-dependent electronic transitions in ethylene

Twisting ethylene about its C=C bond drives the low-lying electronic states through a conical
intersection and induces ground-state degeneracy, classic multireference features. Two quantum algorithms are used to treat this challenging chemistry problem here:

- **DOS-QPE** (density-of-states quantum phase estimation) samples the density of states from a mixed probe state ([arXiv:2510.14744](https://arxiv.org/abs/2510.14744)).
- **QC-QMC** (quantum computing quantum Monte Carlo) refines a VQE reference by imaginary-time walker dynamics ([arXiv:2603.25582](https://arxiv.org/abs/2603.25582)).

Exact diagonalization (full CI) of each active-space Hamiltonian is the reference throughout.

In [ ]:
import contextlib
import io

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pyscf import fci, gto, mcscf, scf

from qarp.operators import JordanWigner, FullyCommuting
from qarp.operators.pyscf import active_space_from_mf
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.ucc import ucc_singles_and_doubles
from qarp.operators.functions import eigenspectrum
from qarp.blocks import TrotterBlock, MappedONVStateBlock, TrotterAnsatzBlock, CompositeBlock
from qarp.algorithms import (
    DOSQPE, VQE, MonteCarlo, StateVector, WalkerState, generate_states_new_basis,
)
from qarp.endianness import bits_to_label
from qarp.engines import QarpEngine
from qarp.optimizers import ScipyOptimizer

## Geometry

Planar ethylene, optimized at the CCSD(T)/cc-pVDZ level. Every
structure below is a rigid rotation of this minimum geometry: a twist of one methylene about the C=C axis, then an
out-of-plane pyramidal flap of that same CH₂.

In [ ]:
_GEOM = np.array([
    [0.0,  0.000000,  0.673547],   # C1
    [0.0,  0.000000, -0.673547],   # C2
    [0.0,  0.934390,  1.246660],   # H on C1
    [0.0, -0.934390,  1.246660],   # H on C1
    [0.0,  0.934390, -1.246660],   # H on C2
    [0.0, -0.934390, -1.246660],   # H on C2
])
_SYM = ["C", "C", "H", "H", "H", "H"]


def build_geometry(twist_deg=0.0, pyr_deg=0.0):
    """Twist the C1 methylene about the C=C axis, then flap it out of plane."""
    g = _GEOM.copy()
    c1 = g[0]

    t = np.radians(twist_deg)
    rz = np.array([[np.cos(t), -np.sin(t), 0.0],
                   [np.sin(t),  np.cos(t), 0.0],
                   [0.0,        0.0,       1.0]])
    for i in (2, 3):
        g[i] = rz @ (g[i] - c1) + c1

    axis = g[2] - g[3]
    axis = axis / np.linalg.norm(axis)          # flap about the H-H axis through C1
    a = np.radians(pyr_deg)
    k = np.array([[0.0, -axis[2], axis[1]],
                  [axis[2], 0.0, -axis[0]],
                  [-axis[1], axis[0], 0.0]])
    rot = np.eye(3) + np.sin(a) * k + (1.0 - np.cos(a)) * (k @ k)
    for i in (2, 3):
        g[i] = rot @ (g[i] - c1) + c1

    return "\n".join(f"{s} {r[0]:.6f} {r[1]:.6f} {r[2]:.6f}" for s, r in zip(_SYM, g, strict=True))


def active_space_hamiltonian(twist_deg, pyr_deg, n_electrons, n_orbitals, basis="cc-pvdz"):
    """Qubit Hamiltonian and reference ONV for a frozen-core CAS of ethylene."""
    mol = gto.M(atom=build_geometry(twist_deg, pyr_deg), basis=basis, verbose=0)
    mf = scf.RHF(mol).run()
    integrals, onv = active_space_from_mf(mf, n_electrons, n_orbitals)
    qham = JordanWigner().encode_operator(restricted_integrals_to_fermion_operator(*integrals))
    return qham, onv

## 1. DOS-QPE across the conical intersection

At a fixed 90° twist, flapping one CH₂ out of plane by the angle **α** sweeps the molecule through
an S₀/S₁ conical intersection. The active space is CAS(2,2) → 4 qubits. DOS-QPE phase-estimates
$U = e^{+iH\tau}$ against a Dicke $|D_{4,2}\rangle$ probe — a uniform mixture over the two-electron
sector — so its sampled distribution peaks at the eigenphases of the triplet **T** and the two
lowest singlets **S₀**, **S₁**.

In [ ]:
def casci_2e2o(twist_deg, pyr_deg):
    """Exact CAS(2,2) levels -> (T, covalent singlet, low ionic, high ionic).

    The two lowest singlets cross along this coordinate, so sorting them by energy
    would swap their identity at the crossing.  They are separated by character
    instead: in the 2x2 CI vector the open-shell (covalent) weight is
    (c01+c10)^2/2, the closed-shell (ionic) weight sits on the diagonal.
    """
    mol = gto.M(atom=build_geometry(twist_deg, pyr_deg), basis="cc-pvdz", verbose=0)
    mf = scf.RHF(mol).run()
    mc = mcscf.CASCI(mf, 2, 2)
    mc.fcisolver.nroots = 4
    mc.verbose = 0
    energies = mc.kernel()[0]
    spins = [mc.fcisolver.spin_square(c, 2, 2)[0] for c in mc.ci]

    triplet = min(e for e, s in zip(energies, spins, strict=True) if s >= 1.5)
    singlets = [(e, np.asarray(c).reshape(2, 2))
                for e, s, c in zip(energies, spins, mc.ci, strict=True) if s < 1.5]
    i_cov = int(np.argmax([0.5 * (v[0, 1] + v[1, 0]) ** 2 for _, v in singlets]))
    ionic = sorted(e for j, (e, _) in enumerate(singlets) if j != i_cov)
    return triplet, singlets[i_cov][0], ionic[0], ionic[1]


alpha_grid = np.arange(50, 96, 3)
levels = np.array([casci_2e2o(90.0, a) for a in alpha_grid])

# One global rescaling puts every eigenphase in [0,1) across the whole scan.
E0, E1 = levels.min(), levels.max()


def scale(e):
    return (e - E0) / (E1 - E0) * 0.9 + 0.05


# Label the crossing singlets by which starts lower, then let character carry it across.
COVALENT_IS_S0 = bool(levels[0, 1] < levels[0, 2])


def as_t_s0_s1_s2(vals):
    t, cov, ion_lo, ion_hi = vals
    return (t, cov, ion_lo, ion_hi) if COVALENT_IS_S0 else (t, ion_lo, cov, ion_hi)


tracked = np.array([as_t_s0_s1_s2(v) for v in levels])

fig, ax = plt.subplots(figsize=(5.4, 4.0))
for col, name, color in [(0, "T", "tab:green"), (1, r"S$_0$", "tab:orange"),
                         (2, r"S$_1$", "tab:purple")]:
    ax.plot(alpha_grid, scale(tracked[:, col]), color=color, lw=1.8, label=name)
for a in (65, 68, 71, 74):
    ax.axvline(a, color="0.8", lw=0.8, ls="--", zorder=0)
ax.set_xlabel(r"pyramidalization angle $\alpha$ [deg]")
ax.set_ylabel("phase (scaled eigenvalue)")
ax.set_title("Ethylene CAS(2,2) low-lying states")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

i_ci = int(np.argmin(np.abs(tracked[:, 2] - tracked[:, 1])))
print(f"S0/S1 gap is smallest at alpha = {alpha_grid[i_ci]} deg -- the conical intersection")

Sampling the density of states at four angles across the intersection: away from it the three
eigenphases are resolved, and at α = 68° the two singlet peaks merge.

In [ ]:
def dos_histogram(dosqpe):
    """Sampler keys are LSB-first bit tuples (qubit q -> key[q]); pack them to an integer."""
    heights = np.zeros(len(dosqpe.freqs))
    for key, p in dosqpe.distribution.items():
        bits = key if isinstance(key, (tuple, list)) else [int(c) for c in key]
        heights[sum(int(b) << i for i, b in enumerate(bits))] += p
    return dosqpe.freqs, heights


N_ANCILLA = 8
ALPHAS = (65, 68, 71, 74)

results = []
for alpha in ALPHAS:
    qham, _ = active_space_hamiltonian(90.0, alpha, 2, 2)
    # Rescale H with the same global window, so the phases DOS-QPE reads are scale(E).
    scaled = (qham - float(E0)) * (0.9 / (E1 - E0)) + 0.05
    # TrotterBlock builds exp(-iH*time); pass -time so U = exp(+iH*time) and phi = E.
    unitary = TrotterBlock(operator=scaled, n_qubits=4, steps=2, time=-2 * np.pi, order=2).build()

    dosqpe = DOSQPE(unitary, N_ANCILLA, hamming_weight=2, engine=QarpEngine(seed=7)).build()
    dosqpe.run()
    x, heights = dos_histogram(dosqpe)
    t, s0, s1, s2 = tracked[np.flatnonzero(alpha_grid == alpha)[0]]
    results.append((alpha, x, heights, t, s0, s1, s2))

    for e in (t, s0, s1):
        assert heights[np.abs(x - scale(e)) <= 2 / 2 ** N_ANCILLA].sum() > 0.02

styles = [("tab:green", "T"), ("tab:orange", r"S$_0$"),
          ("tab:purple", r"S$_1$"), ("0.45", r"S$_2$")]

# The spectrum lives in two tight clusters ~0.65 apart, hence the broken x-axis.
low = [scale(v) for (_, _, _, t, s0, s1, _) in results for v in (t, s0, s1)]
high = [scale(r[6]) for r in results]
lo_win = (min(low) - 0.015, max(low) + 0.015)
hi_win = (min(high) - 0.015, max(high) + 0.015)

fig = plt.figure(figsize=(10.0, 6.0))
outer = fig.add_gridspec(2, 2, hspace=0.60, wspace=0.30)
first = None
for k, (alpha, x, heights, t, s0, s1, s2) in enumerate(results):
    inner = outer[k // 2, k % 2].subgridspec(1, 2, width_ratios=[3, 1], wspace=0.10)
    ax_lo = fig.add_subplot(inner[0], sharey=first)
    ax_hi = fig.add_subplot(inner[1], sharey=ax_lo)
    first = first or ax_lo

    keep = heights > 1e-4
    for ax, win in ((ax_lo, lo_win), (ax_hi, hi_win)):
        for e, (color, _) in zip((t, s0, s1, s2), styles, strict=True):
            ax.axvline(scale(e), color=color, ls="--", lw=1.1, alpha=0.9, zorder=1)
        # Stems, not bars: one bin is 1/256 wide and has no readable width as a bar.
        ax.vlines(x[keep], 0.0, heights[keep], color="0.15", lw=1.0, zorder=3)
        ax.set_xlim(*win)
        ax.tick_params(labelsize=8)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))

    ax_hi.xaxis.set_major_locator(MaxNLocator(nbins=2, steps=[1, 2, 5, 10]))
    ax_lo.spines["right"].set_visible(False)
    ax_hi.spines["left"].set_visible(False)
    ax_hi.tick_params(left=False, labelleft=False)

    d = 0.015
    kw = dict(transform=ax_lo.transAxes, color="0.4", clip_on=False, lw=0.9)
    ax_lo.plot((1 - d, 1 + d), (-d, d), **kw)
    ax_lo.plot((1 - d, 1 + d), (1 - d, 1 + d), **kw)
    kw["transform"] = ax_hi.transAxes
    ax_hi.plot((-3 * d, 3 * d), (-d, d), **kw)
    ax_hi.plot((-3 * d, 3 * d), (1 - d, 1 + d), **kw)

    # 0.67 in ax_lo coords is the centre of the 3:1 pair.
    ax_lo.set_title(rf"$\alpha = {alpha}^\circ$", fontsize=10, x=0.67)
    if k // 2 == 1:
        ax_lo.set_xlabel("phase", fontsize=9)
        ax_lo.xaxis.set_label_coords(0.67, -0.22)
    if k % 2 == 0:
        ax_lo.set_ylabel("probability", fontsize=9)

handles = [plt.Line2D([], [], color=c, ls="--", lw=1.1, label=n) for c, n in styles]
fig.legend(handles=handles, frameon=False, fontsize=9, ncol=4,
           loc="lower center", bbox_to_anchor=(0.5, -0.02))
fig.suptitle(r"DOS-QPE distributions, probe $|D_{4,2}\rangle$", y=0.98)
plt.show()

## 2. QC-QMC along the torsion scan

The full 0°→90° H–C–C–H twist in a **CAS(2,3) → 6-qubit** active space, in a STO-3G basis. Twisted
ethylene is a diradical, so RHF has two solutions at the perpendicular geometry: the symmetric one is
pinned below, and its active-space spectrum is checked against pyscf CASCI on the same orbitals. At 90°
the ground state is a **triplet** (three degenerate levels), with the lowest singlet only ~3 mHa above it,
the structure reported in [arXiv:2603.25582](https://arxiv.org/abs/2603.25582).

A short UCCSD **VQE** then supplies the walker basis, and QC-QMC propagates walkers in imaginary
time to refine the ground-state energy, using loose QMC settings of 500 walkers and
10 trajectories over 100 steps of δτ = 0.1. The VQE is limited to 20 iterations on purpose, and QMC is
responsible for refining the poorly converged result. With these settings QC-QMC recovers exact
diagonalization to under 1 mHa at the three near-planar geometries; at 67.5° and 90°, where the
multireference character is strongest, the residual grows to 8–20 mHa, still an order of magnitude
below the VQE error.


In [ ]:
R_CC, R_CH, A_HCH = 1.324, 1.070, 118.0


def qmc_geometry(torsion_deg):
    """Rigid H-C-C-H torsion: 0 deg planar, 90 deg perpendicular."""
    z = R_CC / 2
    a = np.radians(A_HCH / 2)
    hz, hy = R_CH * np.cos(a), R_CH * np.sin(a)
    t = np.radians(torsion_deg)
    atoms = [("C", (0.0, 0.0, z)), ("C", (0.0, 0.0, -z)),
             ("H", (hy * np.sin(t), hy * np.cos(t), z + hz)),
             ("H", (-hy * np.sin(t), -hy * np.cos(t), z + hz)),
             ("H", (0.0, hy, -z - hz)), ("H", (0.0, -hy, -z - hz))]
    return "\n".join(f"{e} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}" for e, c in atoms)


def qmc_active_space(torsion_deg):
    # Twisted ethylene is a diradical: without point-group symmetry RHF lands on either
    # the symmetric solution or a 58 mHa higher symmetry-broken one, run to run.
    mol = gto.M(atom=qmc_geometry(torsion_deg), basis="sto-3g", symmetry=True, verbose=0)
    mf = scf.RHF(mol).run()
    core = mf.mo_energy[:2]
    assert mf.converged and abs(core[0] - core[1]) < 5e-3, "RHF landed on the symmetry-broken solution"
    integrals, onv = active_space_from_mf(mf, 2, 3)
    qham = JordanWigner().encode_operator(restricted_integrals_to_fermion_operator(*integrals))
    return qham, onv, mf


def lowest_five(qham, n_qubits=6):
    """Five lowest eigenvalues in the two-electron sector.

    The full 2**6 spectrum also carries other particle-number sectors, which the
    physical problem never occupies.
    """
    matrix = qham.sparse_matrix().toarray()
    values, vectors = np.linalg.eigh(matrix)
    number = sum(np.kron(np.eye(2 ** (n_qubits - 1 - k)),
                         np.kron(np.array([[0.0, 0.0], [0.0, 1.0]]), np.eye(2 ** k)))
                 for k in range(n_qubits))
    occupancy = np.round(np.real(np.diag(vectors.conj().T @ number @ vectors))).astype(int)
    order = sorted((i for i in range(2 ** n_qubits) if occupancy[i] == 2),
                   key=lambda i: values[i])[:5]
    return values[order]


qham_90, _, mf_90 = qmc_active_space(90.0)
levels_90 = lowest_five(qham_90)
print("five lowest states at 90 deg (Ha):", np.round(levels_90, 4))

# Independent oracle: pyscf CASCI on the same orbitals, with the symmetry-blind solver
# so every irrep is included.  Its Sz = 0 roots carry the triplet once; the qubit
# spectrum carries all three Ms components.
mc = mcscf.CASCI(mf_90, 3, 2)
mc.fcisolver = fci.direct_spin1.FCI(mf_90.mol)
mc.fcisolver.nroots = 3
mc.verbose = 0
casci_roots = mc.kernel()[0]
assert np.allclose(levels_90[:3], levels_90[0], atol=1e-6)
assert np.allclose(levels_90[[0, 3, 4]], casci_roots, atol=1e-6)
print(f"triplet ground state; lowest singlet {1e3 * (levels_90[3] - levels_90[0]):.1f} mHa above it")


In [ ]:
def qmc_ground_state(torsion_deg, vqe_maxiter=20, n_traj=10, n_walkers=500,
                     total_time=10.0, dt=0.1, seed=1):
    qham, onv, _ = qmc_active_space(torsion_deg)
    exact = float(np.min(eigenspectrum(qham)))

    fucc, symbols = ucc_singles_and_doubles(onv, spin_conserving=True, generalised=True)
    ansatz = CompositeBlock([
        MappedONVStateBlock(onv, JordanWigner()),
        TrotterAnsatzBlock(len(onv), JordanWigner().encode_operator(fucc), symbols,
                           steps=1, time=1, order=1, grouping=FullyCommuting(), imaginary=True),
    ]).build()

    # VQE is stopped early on purpose: a converged reference would leave QC-QMC
    # nothing to correct, and the point here is the correction.
    rng = np.random.default_rng(seed)
    vqe = VQE(operator=qham, ket=ansatz, primitive=StateVector(),
              initial_parameters=rng.random(len(ansatz.symbols)),
              optimizer=ScipyOptimizer(method="COBYLA", options={"maxiter": vqe_maxiter}),
              verbose=False).build()
    # COBYLA at maxiter=20 never reports success, and the library prints that
    # regardless of verbose; the truncation is deliberate, so mute it.
    with contextlib.redirect_stdout(io.StringIO()):
        e_vqe, x_vqe = vqe.run()
    unitary = vqe.final_block.blocks[1].set_symbols(dict(zip(ansatz.symbols, x_vqe, strict=True)))

    states, _, idxs = generate_states_new_basis(unitary, hamming_weight=int(np.sum(onv)))
    basis = [WalkerState(state_data=s, sign=1, label=str(idxs[i])) for i, s in enumerate(states)]

    mc = MonteCarlo(hamiltonian=qham, approx_ground_state_energy=float(e_vqe),
                    total_time=total_time, time_step=dt, initial_walker_count=n_walkers,
                    reference_walker_label=bits_to_label(onv), unitary_block=unitary.build(),
                    walker_basis=basis, shift_damping=0.1, population_threshold=8000,
                    num_trajectories=n_traj, mode="Semiclassical", primitive=StateVector(),
                    verbose=False, seed=seed).build()
    mc.run()
    finals = [np.real(t[0][-1]) for t in mc.energy_estimates_trajectories]
    return exact, float(e_vqe), float(np.mean(finals)), float(np.std(finals))


torsions = [0.0, 22.5, 45.0, 67.5, 90.0]
rows = {}


def run_torsion(tw):
    exact, e_vqe, e_qmc, sd = qmc_ground_state(tw)
    rows[tw] = (tw, exact, e_vqe, e_qmc, sd)
    print(f"torsion {tw:5.1f} deg | exact {exact:.5f} | VQE {e_vqe:.5f} | QC-QMC {e_qmc:.5f}")


In [ ]:
# One torsion per cell: a single QC-QMC run takes minutes, and the notebook CI
# budget is per cell.
run_torsion(0.0)


In [ ]:
run_torsion(22.5)


In [ ]:
run_torsion(45.0)


In [ ]:
run_torsion(67.5)


In [ ]:
run_torsion(90.0)


In [ ]:
rows = np.array([rows[tw] for tw in torsions])
err_vqe = np.abs(rows[:, 2] - rows[:, 1])
err_qmc = np.abs(rows[:, 3] - rows[:, 1])
for tw, ev, eq in zip(torsions, err_vqe, err_qmc, strict=True):
    print(f"torsion {tw:5.1f} deg | VQE error {1e3 * ev:7.2f} mHa | QC-QMC error {1e3 * eq:6.2f} mHa")

# The mixed estimator is not variational: at 0 deg QC-QMC lands slightly below exact.
assert np.all(err_qmc < 0.2 * err_vqe)
assert np.all(err_qmc[:3] < 2e-3)

fig, ax = plt.subplots(figsize=(6.0, 4.4))

ax.plot(rows[:, 0], rows[:, 1], "-", color="0.3", lw=1.8, label="exact diagonalization")
ax.plot(rows[:, 0], rows[:, 2], "o", color="tab:red", ms=7, label="VQE")
ax.plot(rows[:, 0], rows[:, 3], "s", color="tab:blue", ms=7, label="QC-QMC")
ax.ticklabel_format(axis="y", useOffset=False)
ax.set_xlabel("H-C-C-H torsion [deg]")
ax.set_ylabel("ground-state energy [Ha]")
ax.set_title("Ethylene CAS(2,3) ground state")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
